# Visualisation des resultats d'analyse Spark

Ce notebook lit directement les sorties Spark depuis `results/spark/*`.

## Structure attendue

- `results/spark/analysis1/part-*.csv`
- `results/spark/analysis2/part-*.csv`
- `results/spark/analysis3/part-*.csv`
- `results/spark/analysis4/part-*.csv`
- `results/spark/analysis5/part-*.csv`

Depuis ce notebook (`notebook/`), le chemin relatif utilise est `../results/spark/...`.

## Option export HDFS (si besoin)

```bash
docker exec hadoop-master hdfs dfs -get /energy/output/spark /root/results/
docker cp hadoop-master:/root/results ./results_local/
```

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# Chargement auto des sorties Spark depuis ../results/spark/*
# ============================================================

def first_part_csv(pattern):
    matches = sorted(glob.glob(pattern))
    if not matches:
        raise FileNotFoundError(f'Aucun fichier trouve pour: {pattern}')
    return matches[0]

path_a1 = first_part_csv('../results/spark/analysis1/part-*.csv')
path_a2 = first_part_csv('../results/spark/analysis2/part-*.csv')
path_a3 = first_part_csv('../results/spark/analysis3/part-*.csv')
path_a4 = first_part_csv('../results/spark/analysis4/part-*.csv')

print('Fichiers utilises:')
print('-', path_a1)
print('-', path_a2)
print('-', path_a3)
print('-', path_a4)

# analysis1: building_type, avg_site_eui, building_count
eui_df = pd.read_csv(
    path_a1,
    header=None,
    names=['building_type', 'avg_site_eui', 'building_count']
)

# analysis2: community, total_ghg (et parfois colonnes supplementaires)
ghg_raw = pd.read_csv(path_a2, header=None)
if ghg_raw.shape[1] >= 4:
    ghg_df = ghg_raw.iloc[:, :4].copy()
    ghg_df.columns = ['community', 'total_ghg', 'avg_ghg', 'building_count']
else:
    ghg_df = ghg_raw.iloc[:, :2].copy()
    ghg_df.columns = ['community', 'total_ghg']
    ghg_df['avg_ghg'] = np.nan
    ghg_df['building_count'] = np.nan

# analysis3: year, avg_star (et parfois min/max/count)
star_raw = pd.read_csv(path_a3, header=None)
if star_raw.shape[1] >= 5:
    star_df = star_raw.iloc[:, :5].copy()
    star_df.columns = ['year', 'avg_star', 'min_star', 'max_star', 'count']
else:
    star_df = star_raw.iloc[:, :2].copy()
    star_df.columns = ['year', 'avg_star']
    star_df['min_star'] = star_df['avg_star']
    star_df['max_star'] = star_df['avg_star']
    star_df['count'] = np.nan

# analysis4: top emitters (fichier large). On prend les colonnes utiles.
top_raw = pd.read_csv(path_a4, header=None)
top10_df = pd.DataFrame({
    'name': top_raw.iloc[:, 2],
    'type': top_raw.iloc[:, 9],
    'community': top_raw.iloc[:, 8],
    'total_ghg': top_raw.iloc[:, 34],
    'site_eui': top_raw.iloc[:, 33],
})

# Conversions numeriques (robuste aux virgules dans les valeurs)
eui_df['avg_site_eui'] = pd.to_numeric(eui_df['avg_site_eui'], errors='coerce')
eui_df['building_count'] = pd.to_numeric(eui_df['building_count'], errors='coerce')
ghg_df['total_ghg'] = pd.to_numeric(ghg_df['total_ghg'], errors='coerce')
star_df['year'] = pd.to_numeric(star_df['year'], errors='coerce')
star_df['avg_star'] = pd.to_numeric(star_df['avg_star'], errors='coerce')
star_df['min_star'] = pd.to_numeric(star_df['min_star'], errors='coerce')
star_df['max_star'] = pd.to_numeric(star_df['max_star'], errors='coerce')
top10_df['total_ghg'] = pd.to_numeric(top10_df['total_ghg'], errors='coerce')
top10_df['site_eui'] = pd.to_numeric(top10_df['site_eui'], errors='coerce')

# Nettoyage rapide des lignes invalides
star_df = star_df.dropna(subset=['year', 'avg_star', 'min_star', 'max_star'])
eui_df = eui_df.dropna(subset=['building_type', 'avg_site_eui', 'building_count'])
ghg_df = ghg_df.dropna(subset=['community', 'total_ghg'])
top10_df = top10_df.dropna(subset=['name', 'total_ghg'])

# ============================================================
# FIGURE 1 — Average Site EUI by Building Type (Top 15)
# ============================================================
fig, ax = plt.subplots(figsize=(14, 8))
top15_eui = eui_df.nlargest(15, 'avg_site_eui')
bars = ax.barh(
    top15_eui['building_type'],
    top15_eui['avg_site_eui'],
    color='steelblue',
    edgecolor='white'
)
ax.set_xlabel('Average Site EUI (kBtu/sq ft)', fontsize=12)
ax.set_title(
    'Top 15 Building Types by Average Site Energy Use Intensity\nChicago Energy Benchmarking',
    fontsize=14,
    fontweight='bold'
)
ax.invert_yaxis()
for bar, val in zip(bars, top15_eui['avg_site_eui']):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f'{val:.1f}',
        va='center',
        fontsize=9
    )
plt.tight_layout()
plt.savefig('fig1_eui_by_type.png', dpi=150)
plt.show()

# ============================================================
# FIGURE 2 — Top 15 Community Areas by Total GHG Emissions
# ============================================================
fig, ax = plt.subplots(figsize=(14, 8))
top15_ghg = ghg_df.nlargest(15, 'total_ghg')
colors = plt.cm.YlOrRd(np.linspace(0.3, 1.0, len(top15_ghg)))
bars = ax.barh(
    top15_ghg['community'],
    top15_ghg['total_ghg'],
    color=colors,
    edgecolor='white'
)
ax.set_xlabel('Total GHG Emissions (Metric Tons CO2e)', fontsize=12)
ax.set_title(
    'Top 15 Community Areas by Total GHG Emissions\nChicago Energy Benchmarking',
    fontsize=14,
    fontweight='bold'
)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('fig2_ghg_by_community.png', dpi=150)
plt.show()

# ============================================================
# FIGURE 3 — ENERGY STAR Score Evolution by Year
# ============================================================
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(
    star_df['year'],
    star_df['avg_star'],
    'o-',
    color='green',
    linewidth=2,
    markersize=8,
    label='Average Score'
)
ax.fill_between(
    star_df['year'],
    star_df['min_star'],
    star_df['max_star'],
    alpha=0.2,
    color='green',
    label='Min-Max Range'
)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('ENERGY STAR Score', fontsize=12)
ax.set_title(
    'ENERGY STAR Score Evolution Over Years\nChicago Energy Benchmarking',
    fontsize=14,
    fontweight='bold'
)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig3_star_by_year.png', dpi=150)
plt.show()


# ============================================================
# FIGURE 5 — Building Count by Type (Pie Chart)
# ============================================================
fig, ax = plt.subplots(figsize=(12, 8))
top8 = eui_df.nlargest(8, 'building_count')
others_count = eui_df['building_count'].sum() - top8['building_count'].sum()
labels = list(top8['building_type']) + ['Others']
sizes = list(top8['building_count']) + [others_count]
colors_pie = plt.cm.Set3(np.linspace(0, 1, len(labels)))
wedges, texts, autotexts = ax.pie(
    sizes,
    labels=labels,
    autopct='%1.1f%%',
    colors=colors_pie,
    startangle=90
)
ax.set_title(
    'Distribution of Buildings by Type\nChicago Energy Benchmarking',
    fontsize=14,
    fontweight='bold'
)
plt.tight_layout()
plt.savefig('fig5_building_type_distribution.png', dpi=150)
plt.show()

print('All visualizations saved successfully!')